## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab) dan Bab III tahap 10 (GPU **T4**). `REVISI_30OKT.md` R1 (data CSV), R2 (folder kerja `MyDrive/skripsi_wondr_v4/`; `skripsi_wondr_v3/` tidak disentuh), R11 (seluruh pelatihan di T4). `CLAUDE.md` Bagian 3.1, 3.3, 3.4.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v4")    # R2: folder v4; skripsi_wondr_v3 tidak disentuh
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])